# 04: Geometry

The structure module needs a consistent way to describe where each residue is and how it is oriented. This notebook introduces the required tools: rotations, translations, rigid transforms, local residue frames, and rigid-alignment RMSD.

**Read alongside:** `../src/af2_from_scratch/geometry.py`

## Stage map

```text
quaternion q ----------> rotation matrix R
                                  |
translation t --------------------+--> rigid transform T
                                                    |
                                    local point ----+----> global point
                                                    |
                                             inverse transform
                                                    |
                                           recover local point

backbone atoms N, C-alpha, C --> one local frame per residue
predicted and target points --> rigid alignment --> Kabsch RMSD
```

Geometry is a toolbox used by the structure module and geometric losses. It is not another neural-network stage.

In [ ]:
import sys

import torch

sys.path.insert(0, "../src")
torch.manual_seed(0)

## 1. Move between local and global coordinates

A rotation changes orientation without changing distances. A translation changes position. Together they form a **rigid transform**:

```text
global_point = R @ local_point + t
```

A quaternion is a compact way to represent the rotation. `quat_to_rot` normalizes it and converts it into a 3 by 3 rotation matrix `R`.

`make_T` stores `R` and `t` in one 4 by 4 matrix. `apply` moves a local point into global coordinates, while `invert` constructs the transform that moves it back.

In [ ]:
from af2_from_scratch.geometry import apply, invert, make_T, quat_to_rot

rotation = quat_to_rot(torch.tensor([1.0, 0.3, -0.2, 0.1]))
translation = torch.tensor([1.0, 2.0, 3.0])
transform = make_T(rotation, translation)
local_point = torch.tensor([0.0, 0.0, 1.5])
global_point = apply(transform, local_point)
recovered_point = apply(invert(transform), global_point)

print("local point:    ", local_point)
print("global point:   ", global_point)
print("recovered point:", recovered_point)
print("round trip works:", torch.allclose(recovered_point, local_point))

## 2. Give every residue a local frame

A local frame is a small coordinate system attached to one residue. `frames_from_backbone` builds it from three backbone atoms:

1. Put the origin at C-alpha.
2. Point the first axis from C-alpha toward C.
3. Start the second axis toward N, then remove its component along the first axis.
4. Obtain the third axis with a cross product.

The result is an **orthonormal frame**: its axes have unit length, are perpendicular, and form a right-handed coordinate system.

In [ ]:
from af2_from_scratch.geometry import frames_from_backbone

n_atoms = torch.tensor([[0.0, 1.0, 0.0], [1.0, 1.0, 0.0]])
ca_atoms = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
c_atoms = torch.tensor([[1.0, 0.0, 0.0], [2.0, 0.0, 0.0]])
frames = frames_from_backbone(n_atoms, ca_atoms, c_atoms)
rotations = frames[..., :3, :3]
identity = torch.eye(3).expand_as(rotations)

print("residue frames:", tuple(frames.shape))
print("origins equal C-alpha:", torch.allclose(frames[..., :3, 3], ca_atoms))
print(
    "axes are orthonormal:",
    torch.allclose(rotations.transpose(-1, -2) @ rotations, identity),
)

## 3. Compare folds without penalizing global motion

The same fold can be stored at any global position and orientation. Raw RMSD penalizes these harmless differences.

**Kabsch alignment** first finds the rigid rotation and translation that best align two point clouds. Kabsch RMSD then measures the remaining structural difference. This makes it useful for evaluation; training later uses local-frame losses for a related invariance.

In [ ]:
from af2_from_scratch.geometry import kabsch_rmsd

points = torch.tensor(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.0, 0.0],
        [1.0, 1.0, 0.0],
        [0.2, 0.4, 1.0],
    ]
)
move = make_T(
    quat_to_rot(torch.tensor([1.0, 0.0, 0.0, 0.5])),
    torch.tensor([4.0, -2.0, 3.0]),
)
moved_points = apply(move, points)
raw_rmsd = (points - moved_points).square().sum(-1).mean().sqrt()
aligned_rmsd = kabsch_rmsd(points, moved_points)

print(f"raw RMSD:     {raw_rmsd.item():.4f}")
print(f"Kabsch RMSD:  {aligned_rmsd.item():.4f}")

**Recap:** Rigid transforms move points between local and global coordinates. Residue frames attach those coordinates to the backbone, and Kabsch alignment removes arbitrary global motion during evaluation.

Next, `05_structure_module.ipynb` uses residue frames inside invariant point attention.